[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/09_naive_bayes/04_why_naive_works_and_fails.ipynb)

# Bài 04 — Vì sao "ngây thơ" vẫn hiệu quả, và khi nào thất bại?

**🎯 Mục tiêu**
- Hiểu nghịch lý: giả định độc lập gần như luôn sai, nhưng Naive Bayes vẫn **phân loại** tốt — còn **xác suất** thì không đáng tin.
- Naive Bayes vs Logistic Regression: ai thắng khi ít dữ liệu, khi nhiều dữ liệu?
- Failure case: feature tương quan (xác suất quá tự tin), tương tác (XOR), phân phối sai giả định.
- Hoàn thiện template model cho Naive Bayes.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/09_naive_bayes

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.naive_bayes import GaussianNB

from ml_course.visualization import plot_decision_boundary, set_style

set_style()
rng = np.random.default_rng(0)

## 1. Feature trùng lặp → xác suất quá tự tin

Thí nghiệm: một feature có ích $x$, rồi **nhân bản** nó $r$ lần (cực đoan của tương quan). Naive Bayes coi mỗi bản sao là một bằng chứng **độc lập** → cộng log-likelihood $r$ lần → posterior bị đẩy về 0/1.

In [ ]:
n = 4000
y = rng.integers(0, 2, n)
x = rng.normal(loc=y * 1.0, scale=1.0)                    # một feature, hai lớp chồng lấn
rows = []
for r in [1, 2, 5, 10]:
    X = np.repeat(x[:, None], r, axis=1) + rng.normal(0, 0.05, (n, r))   # r bản sao (gần như giống hệt)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.5, random_state=0)
    p = GaussianNB().fit(X_tr, y_tr).predict_proba(X_te)[:, 1]
    rows.append((r, p))
    print(f"{r:2d} bản sao: accuracy = {accuracy_score(y_te, p >= 0.5):.3f} | log-loss = {log_loss(y_te, p):.3f} | "
          f"% dự đoán với xác suất > 0.99 hoặc < 0.01: {np.mean((p > 0.99) | (p < 0.01)):.0%}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for r, p in rows:
    frac_pos, mean_pred = calibration_curve(y_te, p, n_bins=10, strategy="quantile")
    ax.plot(mean_pred, frac_pos, "o-", label=f"{r} bản sao")
ax.plot([0, 1], [0, 1], "k--", label="calibrated hoàn hảo")
ax.set_xlabel("xác suất dự đoán")
ax.set_ylabel("tỉ lệ thực tế lớp 1")
ax.set_title("Reliability diagram: càng nhiều bản sao càng quá tự tin")
ax.legend()
plt.show()

**Accuracy gần như không đổi**, nhưng log-loss tăng vọt và xác suất bị đẩy về hai đầu. Đây là bản chất của "vì sao naive vẫn hiệu quả" (Domingos & Pazzani, 1997): phân loại chỉ cần **thứ tự** posterior đúng ($\arg\max$), không cần giá trị chính xác. Tương quan làm sai độ lớn posterior nhưng thường không đảo thứ tự.

> ⚠️ Hệ quả thực tế: đừng dùng `predict_proba` của Naive Bayes như xác suất thật (để tính chi phí, rủi ro) — hãy **calibrate** (chương 37):

In [ ]:
X = np.repeat(x[:, None], 10, axis=1) + rng.normal(0, 0.05, (n, 10))
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.5, random_state=0)
raw = GaussianNB().fit(X_tr, y_tr).predict_proba(X_te)[:, 1]
cal = CalibratedClassifierCV(GaussianNB(), method="isotonic", cv=5).fit(X_tr, y_tr).predict_proba(X_te)[:, 1]
print(f"log-loss: NB gốc = {log_loss(y_te, raw):.3f} | NB + isotonic calibration = {log_loss(y_te, cal):.3f}")

## 2. Naive Bayes vs Logistic Regression: ít dữ liệu vs nhiều dữ liệu

Ng & Jordan (2002): model sinh (NB) tiến tới lỗi tiệm cận của nó **nhanh hơn** — cần ít mẫu hơn — nhưng khi giả định độc lập sai, lỗi tiệm cận của model phân biệt (LR) **thấp hơn**. Hai learning curve:
- **(a) Review sản phẩm** (bag-of-words, 3 lớp) — các từ gần như độc lập khi biết lớp;
- **(b) Chữ số viết tay nhị phân hoá** (pixel > 7 → 1) với Bernoulli NB — pixel cạnh nhau **tương quan rất mạnh**, giả định naive sai nặng.

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import BernoulliNB, MultinomialNB

from ml_course.datasets import load_digits_data


def learning_curve_nb_lr(make_xy, sizes, n_rep=20):
    out = {"Naive Bayes": [], "Logistic Regression": []}
    for m in sizes:
        accs = {k: [] for k in out}
        for rep in range(n_rep):
            data = make_xy(m, rep)
            if data is None:                      # mẫu quá nhỏ thiếu lớp → bỏ qua lần lặp này
                continue
            Xs, ys, Xt, yt, nb = data
            accs["Naive Bayes"].append(nb.fit(Xs, ys).score(Xt, yt))
            accs["Logistic Regression"].append(LogisticRegression(max_iter=3000).fit(Xs, ys).score(Xt, yt))
        for k in out:
            out[k].append(np.mean(accs[k]))
    return out


reviews = pd.read_csv("../../data/sample/reviews_vi.csv")
rv_pool, rv_eval = train_test_split(reviews, test_size=0.5, random_state=0, stratify=reviews.sentiment)


def reviews_xy(m, rep):
    sub = rv_pool.sample(m, random_state=rep)
    if sub.sentiment.nunique() < 3:
        return None
    cv = CountVectorizer().fit(sub.text)
    return cv.transform(sub.text), sub.sentiment, cv.transform(rv_eval.text), rv_eval.sentiment, MultinomialNB()


Xd, yd, _ = load_digits_data(as_float=False)
Xd = (Xd > 7).astype(int)
dg_pool, dg_eval, yd_pool, yd_eval = train_test_split(Xd, yd, test_size=0.4, random_state=0, stratify=yd)


def digits_xy(m, rep):
    idx = np.random.default_rng(rep).choice(len(dg_pool), m, replace=False)
    return dg_pool[idx], yd_pool[idx], dg_eval, yd_eval, BernoulliNB()


curves = {"(a) review — từ gần độc lập": ([6, 10, 20, 40, 100, 300], reviews_xy),
          "(b) digits nhị phân — pixel tương quan mạnh": ([20, 50, 100, 200, 500, 1000], digits_xy)}
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, (title, (sizes, make_xy)) in zip(axes, curves.items()):
    res = learning_curve_nb_lr(make_xy, sizes)
    for k, v in res.items():
        ax.semilogx(sizes, v, "o-", label=k)
    ax.set_title(title)
    ax.set_xlabel("số mẫu train")
    ax.set_ylabel("accuracy")
    ax.legend()
    print(title)
    for m, a, b in zip(sizes, res["Naive Bayes"], res["Logistic Regression"]):
        print(f"   m = {m:5d}: NB = {a:.3f} | LR = {b:.3f}")
plt.show()

- **(a)** Với vài chục review, NB thắng rõ; khi đủ dữ liệu cả hai gần như hoàn hảo.
- **(b)** NB **chững lại sớm** (từ 500 lên 1000 mẫu gần như không tăng) vì pixel tương quan bị "đếm trùng"; LR tiếp tục cải thiện và bỏ xa.

**Quy tắc thực hành:** rất ít dữ liệu, hoặc cần baseline trong vài giây → Naive Bayes; nhiều dữ liệu → model phân biệt (LR, boosting...) thường thắng.

## 3. Failure case: tương tác giữa các feature (XOR)

Nếu nhãn chỉ phụ thuộc vào **sự kết hợp** của các feature, mỗi feature riêng lẻ không mang thông tin gì: $P(x_1 \mid y)$ giống nhau cho cả hai lớp → Naive Bayes chỉ đoán mò.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

from ml_course.datasets import make_classification_2d

Xx, yx = make_classification_2d("xor", n=600, noise=0.25, seed=0)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for ax, (name, model) in zip(axes, [("Gaussian NB", GaussianNB()), ("KNN (k=15)", KNeighborsClassifier(15))]):
    acc = cross_val_score(model, Xx, yx, cv=5).mean()
    plot_decision_boundary(model.fit(Xx, yx), Xx, yx, ax=ax)
    ax.set_title(f"{name}: accuracy CV = {acc:.2f}")
plt.show()

## 4. Failure case: phân phối sai giả định

Gaussian NB giả định mỗi feature trong mỗi lớp là **một** đường chuông. Dữ liệu: lớp 0 có **ba** cụm (quanh −3, 0, 3), lớp 1 có **hai** cụm (quanh −1.5, 1.5) — xen kẽ nhau. Một Gaussian cho mỗi lớp không thể mô tả cấu trúc này. **Khắc phục:** rời rạc hoá feature (`KBinsDiscretizer` + `CategoricalNB`), dùng kernel density cho từng feature, hoặc biến đổi feature.

In [ ]:
from sklearn.naive_bayes import CategoricalNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import KBinsDiscretizer

n = 3000
y_b = rng.integers(0, 2, n)
x_b = np.where(y_b == 0, rng.choice([-3.0, 0.0, 3.0], n), rng.choice([-1.5, 1.5], n)) + rng.normal(0, 0.5, n)
X_b = np.c_[x_b, rng.normal(size=n)]                # + 1 feature nhiễu
gauss = cross_val_score(GaussianNB(), X_b, y_b, cv=5).mean()
binned = cross_val_score(make_pipeline(KBinsDiscretizer(n_bins=20, encode="ordinal", strategy="uniform"),
                                       CategoricalNB()), X_b, y_b, cv=5).mean()
print(f"Gaussian NB = {gauss:.3f} | rời rạc hoá (20 bin) + CategoricalNB = {binned:.3f}")

In [ ]:
grid = np.linspace(-6, 6, 400)
for c, color in [(0, "tab:blue"), (1, "tab:orange")]:
    plt.hist(x_b[y_b == c], bins=60, density=True, alpha=0.35, color=color, label=f"lớp {c} (thực tế)")
    m, s = x_b[y_b == c].mean(), x_b[y_b == c].std()
    plt.plot(grid, np.exp(-(grid - m) ** 2 / (2 * s**2)) / np.sqrt(2 * np.pi * s**2), color=color, lw=2,
             label=f"Gaussian NB ước lượng lớp {c}")
plt.legend(fontsize=8)
plt.title("Một Gaussian mỗi lớp không mô tả được phân phối nhiều đỉnh")
plt.show()

Gaussian NB chỉ học được "lớp 1 tập trung hơn lớp 0" (phương sai nhỏ hơn) → dự đoán lớp 1 cho cả vùng giữa, kể cả cụm lớp 0 quanh 0. Histogram theo bin bắt được cấu trúc nhiều đỉnh.

## 5. Template model: Naive Bayes

| Phần | Naive Bayes |
|---|---|
| Problem | phân loại (nhị phân, nhiều lớp); đặc biệt văn bản, spam, baseline nhanh |
| Intuition | mỗi feature là một bằng chứng độc lập; cộng log-bằng chứng với prior |
| Geometry | Multinomial/Bernoulli: tuyến tính trong không gian feature; Gaussian: bậc hai |
| Probability | model sinh: $P(y)\prod_j P(x_j \mid y)$; posterior qua Bayes |
| Objective | maximum likelihood (đếm tần suất / mean-var) + smoothing (≈ MAP với Dirichlet prior) |
| Optimization | nghiệm đóng — không cần lặp |
| Assumptions | độc lập có điều kiện; dạng phân phối của $P(x_j \mid y)$ |
| Complexity | train $O(nd)$ một lượt; dự đoán $O(Cd)$; rất ít bộ nhớ |
| Hyperparameters | `alpha` (Multinomial/Bernoulli), `var_smoothing` (Gaussian), `priors`, `fit_prior` |
| Preprocessing | văn bản: tokenization, n-gram; Gaussian: biến đổi feature lệch; không cần scaling |
| Strengths | rất nhanh, ít dữ liệu vẫn tốt, nhiều chiều (văn bản) tốt, online learning (`partial_fit`), xử lý missing tự nhiên |
| Weaknesses | xác suất không calibrated (quá tự tin); không học tương tác; nhạy với giả định phân phối; feature tương quan bị đếm trùng |
| Interpretation | log-tỉ số $\log\theta_{1w} - \log\theta_{0w}$ cho từng feature |

## 🧠 Kiểm tra nhanh

1. Vì sao Naive Bayes phân loại tốt dù xác suất sai? Khi nào điều đó không còn đúng?
2. Bạn có 50 email gắn nhãn. Naive Bayes hay Logistic Regression? Còn khi có 500 000 email?
3. Vì sao Naive Bayes thất bại hoàn toàn với XOR, trong khi KNN và Decision Tree thì không?
4. Một feature có phân phối hai đỉnh trong một lớp — đề xuất 2 cách xử lý.

## 📌 Tóm tắt

- Giả định độc lập sai → xác suất quá tự tin, nhưng $\arg\max$ thường vẫn đúng → phân loại tốt; calibrate nếu cần xác suất.
- NB thắng khi ít dữ liệu, LR thắng khi đủ dữ liệu (Ng & Jordan).
- Thất bại với tương tác (XOR) và khi phân phối sai giả định.

## 📚 Tài liệu tham khảo

- Domingos & Pazzani (1997) — *On the Optimality of the Simple Bayesian Classifier under Zero-One Loss*
- Ng & Jordan (2002) — *On Discriminative vs. Generative Classifiers: A comparison of logistic regression and naive Bayes*
- Rennie et al. (2003) — *Tackling the Poor Assumptions of Naive Bayes Text Classifiers* (Complement NB)
- Zhang (2004) — *The Optimality of Naive Bayes*